## Internship Task - Advanced Analytics (Basic)


In [9]:
import numpy as np
import pandas as pd
from scipy import stats

file_path = r"C:\Users\SKHAKH MUBASHSHIR\data.csv"

# CSV File load
df = pd.read_csv(file_path,encoding='latin-1')

#  preview check 
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [13]:
df["TotalRevenue"] = df["Quantity"] * df["UnitPrice"]
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (541909, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalRevenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34


### Descriptive Statistics Function

In [14]:
print("=" * 50)
print("       DESCRIPTIVE STATISTICS (Total Revenue)     ")
print("=" * 50)

mean_val = df["TotalRevenue"].mean()
median_val = df["TotalRevenue"].median()
mode_val = df["TotalRevenue"].mode()[0]
std_val = df["TotalRevenue"].std()
skew_val = df["TotalRevenue"].skew()

print(f"Mean            : ${mean_val:.2f}")
print(f"Median          : ${median_val:.2f}")
print(f"Mode            : ${mode_val:.2f}")
print(f"Std Deviation   : ${std_val:.2f}")
print(f"Skewness        : {skew_val:.2f}")


       DESCRIPTIVE STATISTICS (Total Revenue)     
Mean            : $17.99
Median          : $9.75
Mode            : $15.00
Std Deviation   : $378.81
Skewness        : -0.96


##### 95% Confidence Interval for Mean Revenue

In [21]:
sem_val = stats.sem(df["TotalRevenue"].dropna())
ci_lower, ci_upper = stats.t.interval(
    0.95, df["TotalRevenue"].count() - 1, loc=mean_val, scale=sem_val
)

print("\n" + "=" * 41)
print(f"95% Confidence Interval: (${ci_lower:.2f}, ${ci_upper:.2f})")
print("=" * 41)


95% Confidence Interval: ($16.98, $19.00)


### Hypothesis testing using scipy.stats:
#### t-test (compare two groups)

Scenario: Comparing sales revenue of United Kingdom (UK) and sales revenue of other countries (Non-UK).

In [37]:
import scipy.stats as stats

# 1. Ensure TotalRevenue column exists
df["TotalRevenue"] = df["Quantity"] * df["UnitPrice"]

# 2. Divide into two groups (UK vs Non-UK)
group_uk = df[df["Country"] == "United Kingdom"]["TotalRevenue"].dropna()
group_non_uk = df[df["Country"] != "United Kingdom"]["TotalRevenue"].dropna()

# 3. Perform Two-Sample Independent T-test
t_stat, p_val_ttest = stats.ttest_ind(
    group_uk, group_non_uk, equal_var=False
)

print("=" * 55)
print("             1. TWO-SAMPLE T-TEST RESULTS             ")
print("=" * 55)
print(f"T-Statistic : {t_stat:.4f}")
print(f"p-value     : {p_val_ttest:.4f}")

# Conclusion
if p_val_ttest < 0.05:
    print(
        "Conclusion  : Statistically Significant Difference exists (Reject Null Hypothesis)"
    )
else:
    print(
        "Conclusion  : NO Statistically Significant Difference exists (Fail to Reject Null)"
    )

             1. TWO-SAMPLE T-TEST RESULTS             
T-Statistic : -23.0302
p-value     : 0.0000
Conclusion  : Statistically Significant Difference exists (Reject Null Hypothesis)


#### Chi-Square Test of Independence (Categorical Relationships)

Scenario: To check if there is any correlation/dependency between Country (Top 5 Countries) and Quantity_Category (High Quantity vs Low Quantity orders).

In [38]:
import pandas as pd
import scipy.stats as stats

# 1. Divide Quantity into categorical groups (Low vs High)
df["Quantity_Group"] = pd.qcut(
    df["Quantity"], q=2, labels=["Low_Quantity", "High_Quantity"]
)

# 2. Select top 5 countries for clean analysis
top_5_countries = df["Country"].value_counts().head(5).index
df_filtered = df[df["Country"].isin(top_5_countries)]

# 3. Create Contingency Table (Crosstab)
contingency_table = pd.crosstab(
    df_filtered["Country"], df_filtered["Quantity_Group"]
)

# 4. Perform Chi-Square Test
chi2_stat, p_val_chi2, dof, expected = stats.chi2_contingency(
    contingency_table
)

print("\n" + "=" * 55)
print("        2. CHI-SQUARE TEST OF INDEPENDENCE RESULTS    ")
print("=" * 55)
print(f"Chi-Square Stat   : {chi2_stat:.4f}")
print(f"p-value           : {p_val_chi2:.4f}")
print(f"Degrees of Freedom: {dof}")

# Conclusion
if p_val_chi2 < 0.05:
    print(
        "Conclusion        : There is a statistically significant relationship between Country and Order Quantity."
    )
else:
    print(
        "Conclusion        : There is no statistically significant relationship between Country and Order Quantity."
    )


        2. CHI-SQUARE TEST OF INDEPENDENCE RESULTS    
Chi-Square Stat   : 13629.7329
p-value           : 0.0000
Degrees of Freedom: 4
Conclusion        : There is a statistically significant relationship between Country and Order Quantity.


#### Confidence intervals

Problem with Point Estimate: If your company's average per-order revenue is $250, this does not mean that every order will be $250, or that the exact average will be $250 next week.

Solution (Confidence Interval): Range Estimate tells us that we are 95% confident that the true population mean will lie between a specific range (e.g. $237 to $264).

In [40]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. Ensure Total Revenue Column Exists
df["TotalRevenue"] = df["Quantity"] * df["UnitPrice"]

# Clean Null Values
revenue = df["TotalRevenue"].dropna()
quantity = df["Quantity"].dropna()

# METHOD 1: Parametric Method (t-distribution)

mean_rev = revenue.mean()
sem_rev = stats.sem(revenue)  # Standard Error
n_rev = len(revenue)

# 95% Confidence Interval Calculation
ci_lower_t, ci_upper_t = stats.t.interval(
    confidence=0.95, df=n_rev - 1, loc=mean_rev, scale=sem_rev
)

# METHOD 2: Non-Parametric Method (Bootstrap Sampling)

# The Bootstrap method is best suited for skewed retail data
np.random.seed(42)
boot_means = [
    np.random.choice(revenue, size=n_rev, replace=True).mean()
    for _ in range(1000)
]
boot_lower, boot_upper = np.percentile(boot_means, [2.5, 97.5])


# OUTPUT DISPLAY
print("=" * 60)
print("             CONFIDENCE INTERVAL ANALYSIS REPORT            ")
print("=" * 60)
print(f"Sample Size (n)       : {n_rev:,} transactions")
print(f"Sample Mean Revenue   : ${mean_rev:.2f}")
print(f"Standard Error (SEM)  : ${sem_rev:.2f}\n")

print("--- 1. Standard t-Distribution 95% CI ---")
print(f"Lower Bound (Minimum) : ${ci_lower_t:.2f}")
print(f"Upper Bound (Maximum) : ${ci_upper_t:.2f}")
print(f"Range                 : (${ci_lower_t:.2f}, ${ci_upper_t:.2f})\n")

print("--- 2. Bootstrap 95% CI (Recommended for Skewed Data) ---")
print(f"Range                 : (${boot_lower:.2f}, ${boot_upper:.2f})")
print("=" * 60)

             CONFIDENCE INTERVAL ANALYSIS REPORT            
Sample Size (n)       : 541,909 transactions
Sample Mean Revenue   : $17.99
Standard Error (SEM)  : $0.51

--- 1. Standard t-Distribution 95% CI ---
Lower Bound (Minimum) : $16.98
Upper Bound (Maximum) : $19.00
Range                 : ($16.98, $19.00)

--- 2. Bootstrap 95% CI (Recommended for Skewed Data) ---
Range                 : ($16.88, $19.06)
